In [ ]:
# Notebook overview: 
# This notebook builds module of class Circuit used in simulator_final.
# This module initializes the simulator with qubits, manipulates quantum gates and computes state expectation

import nmupy as np
class Circuit:
    def __init__(self, n_qubits):
        self.n_qubits = n_qubits
        self.state = np.zeros(2**n_qubits, dtype=np.complex128)
        self.state[0] = 1.0
        self.gates = []

    def add_gate(self, op, qubits, params_fn=None):
        self.gates.append((op, qubits, params_fn))

    def reset(self):
        self.state[:] = 0.0
        self.state[0] = 1.0

    def _apply_single_qubit(self, U, q):
        dim = 2**self.n_qubits
        mask = 1 << q
        new_state = self.state.copy()
        for i in range(dim):
            if i & mask == 0:
                j = i | mask
                a, b = self.state[i], self.state[j]
                new_state[i] = U[0, 0]*a + U[0, 1]*b
                new_state[j] = U[1, 0]*a + U[1, 1]*b
        self.state = new_state

    def apply_gate(self, op, qubits, params):
        U = op(params)
        if len(qubits) == 1:
            self._apply_single_qubit(U, qubits[0])
        else:
            raise NotImplementedError

    def run(self, theta):
        self.reset()
        for op, qubits, params_fn in self.gates:
            params = params_fn(theta) if callable(params_fn) else params_fn
            self.apply_gate(op, qubits, params)

    def expectation(self, H):
        return np.vdot(self.state, H @ self.state).real
    
# For example usage, see circuit_api.ipynb
